# Module 6: Transformer Based Models

The Transformer architecture ("Attention Is All You Need," 2017) fundamentally changed NLP and is now the foundation of GPT, BERT, Llama, and virtually every modern language model.

This notebook builds self-attention and multi-head attention from scratch, then demonstrates a full transformer block.

## 1. Self-Attention from Scratch

**The core idea:** For each word in a sequence, self-attention computes how much to "attend to" every other word. The sentence "The cat sat on the mat because **it** was tired" — what does "it" refer to? Self-attention learns that "it" should attend strongly to "cat."

**The math:**
- **Q (Query):** What am I looking for?
- **K (Key):** What do I contain?
- **V (Value):** What information do I provide?
- **Attention = softmax(QK^T / sqrt(d)) x V**

Dividing by sqrt(d) prevents the dot products from getting too large (which would push softmax to extremes and kill gradients).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.scale = math.sqrt(embed_dim)

    def forward(self, x, mask=None):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output, attention_weights

x = torch.randn(2, 10, 64)  # batch=2, seq_len=10, embed_dim=64
sa = SelfAttention(64)
out, weights = sa(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\nEach token now contains information from ALL other tokens,")
print(f"weighted by how relevant they are (attention weights sum to 1).")

## 2. Multi-Head Attention

**Why multiple heads?** A single attention head can only focus on one type of relationship at a time. Multiple heads allow the model to simultaneously attend to different aspects:
- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn semantic relationships (pronoun-antecedent)
- Head 3 might learn positional relationships (nearby words)

We split the embedding into `num_heads` chunks, apply attention independently, then concatenate.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        self.scale = math.sqrt(self.head_dim)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.W_q(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / self.scale, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(context), attn

mha = MultiHeadAttention(64, num_heads=4)
out, attn = mha(x)
print(f"4-head attention output: {out.shape}")
print(f"Attention per head: {attn.shape}  (batch, heads, seq, seq)")
print(f"Each head attends independently with {64//4}d representations")

## 3. Full Transformer Block

A transformer block combines:
1. **Multi-head attention** + residual connection + layer norm
2. **Feed-forward network** + residual connection + layer norm

The residual connections (adding the input back) solve the degradation problem — deeper networks can always learn the identity function and "pass through" information unchanged. Layer norm stabilizes training.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        attn_out, _ = self.attention(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x

tb = TransformerBlock(embed_dim=64, num_heads=4, ff_dim=256)
out = tb(x)
print(f"Transformer block: {x.shape} -> {out.shape}")
print(f"\nThis is one layer of BERT/GPT. Stack 12 of these = BERT-base.")
print(f"Stack 96 of these = GPT-3.")

## Key Takeaways

1. **Self-attention** lets every token "look at" every other token — O(n^2) but captures long-range dependencies
2. **Multi-head attention** allows parallel focus on different relationship types
3. **Residual connections + LayerNorm** are essential for training deep transformers
4. **BERT** uses bidirectional attention (sees all tokens) — great for understanding/classification
5. **GPT** uses causal attention (sees only past tokens) — great for generation